# Lesson 5, Part 1

Here's a simple calculator for square roots, 
where *tol*  indicates the maximum tolerance for relative error. 
It usually works well.

In [ ]:
import numpy as np

def easy_sqrt(number,estimate=1,tol=1E-14):
    for n in range(1,100):
        diff = number / estimate - estimate
        estimate += 0.5 * diff
        if abs(diff) < estimate * tol:
            break       
    return estimate,n

for number in np.linspace(1,49,5):
    root,n = easy_sqrt(number)
    print('sqrt of',number,'=',root,'in',n,'iterations')

However, many iterations may be needed. 
Also, estimates for negative numbers never converge.

In [ ]:
import matplotlib.pyplot as plt

def easy_sqrt_display(number,estimate=1,tol=1E-14):
    record = [estimate]
    for n in range(1,100):
        diff = number / estimate - estimate
        estimate += 0.5 * diff
        record.append(estimate)
        if abs(diff) < estimate * tol:
            break
    plt.bar(range(len(record)),record)  # creates bar plot
    return estimate

for number in (9,9000,-2,-4):
    plt.figure()
    plt.title('sqrt of ' + str(number))
    easy_sqrt_display(number)
plt.show()

For fast convergence, the initial estimate needs to have the right order of magnitude.

In [ ]:
for number in np.geomspace(1E-16,1E16,17):  
    root,n = easy_sqrt(number)
    print('sqrt of ',number,'=',round(root,12),'in',n,'iterations')

Here's one way to seed the square root calculator with a better initial estimate.

In [ ]:
def seed_sqrt(number):
    seed = 1
    if number > 1:
        while number >= 100:
            seed *= 10
            number /= 100
        while number >= 9:
            seed *= 3
            number /= 9
    elif 0 < number < 1:         
        while number * 100 <= 1:
            seed /= 10
            number *= 100
        while number * 9 <= 1:
            seed /= 3
            number *= 9
    return seed

number = 6E40
root,n = easy_sqrt(number)
print('sqrt of',number,'=',root,'in',n,'iterations')
root,n = easy_sqrt(number,seed_sqrt(number))
print('With better seed: \nsqrt of',number,'=',root,'in',n,'iterations')

We can seed faster by accessing the sequences of 1s and 0s used to store numbers on computers. 
The math command **frexp** reveals the conversion *number = mantissa* times  *2****exponent*, 
where 0.5<=*mantissa<*1 and *exponent* is an integer. 
The math command **ldexp**  redoes the conversion. As usual, numpy offers a version that inputs and outputs ndarrays.

In [ ]:
import math

numbers = (1E-6,1E-4,1,1E-2,1E2,1E4,1E6,-4,0)
for number in numbers:
    mantissa,exponent = math.frexp(number)
    number2 = math.ldexp(mantissa,exponent)
    print(number,'=',mantissa,'* 2**',exponent,'=',number2)
    
mantissas,exponents = np.frexp(numbers)
numbers2 = np.ldexp(mantissas,exponents)
print('\nnumbers =',numbers,'\nmantissas =',mantissas,'\nexponents =',exponents,'\nnumbers2 =',numbers2)

This code incorporates **frexp**  to make a faster square root calculator
and compares the speed of different methods.

In [ ]:
import time

def easy_sqrt_alt(number,tol=1E-14):
    mantissa,exponent = math.frexp(number)
    half_exponent,remainder = divmod(exponent,2)   # returns whole number quotient and remainder
    if remainder:   # false if remainder==0
        mantissa *= 2
    estimate = (1 + mantissa) / 2
    for n in range(2,80):
        diff = mantissa / estimate - estimate
        estimate += 0.5 * diff
        if abs(diff) < estimate * tol:
            break      
    estimate = math.ldexp(estimate,half_exponent)
    return estimate, n

number,iters = 6E40,1000
tic = time.time()
for iter in range(iters):
    sqrt_num,n = easy_sqrt(number)
toc = time.time()
print('Square root of',number,'using easy_sqrt:\n',sqrt_num,'in',n,'iterations, which takes time',toc-tic)
tic = time.time()
for iter in range(iters):
    sqrt_num,n = easy_sqrt(number,seed_sqrt(number))
toc = time.time()
print('Square root of',number,'using easy_sqrt with seed_sqrt:\n',sqrt_num,'in',n,'iterations, which takes time',toc-tic)
tic = time.time()
for iter in range(iters):
    sqrt_num,n = easy_sqrt_alt(number)
toc = time.time()
print('Square root of',number,'using easy_sqrt_alt:\n',sqrt_num,'in',n,'iterations, which takes time',toc-tic)
tic = time.time()
for iter in range(iters): 
    sqrt_num = math.sqrt(number)
toc = time.time()
print('Square root of',number,'using math.sqrt:\n',sqrt_num,'which takes time',toc-tic)

By making an initial estimate that is relatively close to the true square root, 
easy_sqrt_alt usually converges in 5 iterations and never needs more than 6.

In [ ]:
max_n = sum_n = 0
for number in np.linspace(0.5,2,1000):
    estimate,n = easy_sqrt_alt(number)
    sum_n += n
    if n > max_n:
        max_n = n
print('Maximum number of iterations is',max_n,'and mean number is',sum_n/1000)

The code here drops the interim checks against tolerance. 
However, it is slower than **np.sqrt** (which can work with ndarrays) and much slower than **math.sqrt( )**. 
Built-in commands typically incorporate more simplifications and speedups than are possible in ordinary Python code.

In [ ]:
def easy_sqrt_alt2(number):
    mantissa,exponent = math.frexp(number)
    half_exponent,remainder = divmod(exponent,2)
    if remainder:
        mantissa *= 2
    estimate = (1 + mantissa) / 2
    for n in range(2,7):
        diff = mantissa / estimate - estimate
        estimate += 0.5 * diff
    return math.ldexp(estimate,half_exponent)

number,iters = 6E40,10000
tic = time.time()
for iter in range(iters):
    sqrt_num = easy_sqrt_alt2(number)
toc = time.time()
print('Square root of',number,'using easy_sqrt_alt2 takes time',toc-tic)
tic = time.time()
for iter in range(iters):
    sqrt_num = np.sqrt(number)
toc = time.time()
print('Square root of',number,'using np.sqrt takes time',toc-tic)
tic = time.time()
for iter in range(iters):
    sqrt_num = math.sqrt(number)
toc = time.time()
print('Square root of',number,'using math.sqrt takes time',toc-tic)

Here is analogous code for 1\sqrt(*number*) using the algorithm described in the text. 
Since convergence nearly always takes 5-6 operations, we won't bother checking within the loop.

In [ ]:
def reciprocal_sqrt(number):
    if number <= 0:
        return math.nan  # stands for "not a number"
    mantissa,exponent = math.frexp(number)
    half_exponent = exponent // 2
    if exponent > 2 * half_exponent:
        mantissa *= 2
    estimate = 1
    for n in range(6):
        mult = 1.5 - 0.5 * mantissa * estimate * estimate
        estimate *= mult
    return math.ldexp(estimate,-half_exponent)

# demo
for number in (1/49,1/36,1/25,1/16,1/9,1/4,4,9,16,25,36,49,-1):
    print('Reciprocal sqrt of',number,'=',reciprocal_sqrt(number))

Here is code for calculating 1/*number* using multiplication.

In [ ]:
def reciprocal_by_mult(number):
    if number==0:
        return math.inf       # stands for "infinity"
    if number==math.inf:
        return 0
    mantissa,exponent = math.frexp(number)
    y = 1 - abs(mantissa)
    estimate = 1 + y 
    for n in range(6):
        y *= y
        estimate *= 1 + y 
    if mantissa < 0:
        estimate = -estimate
    return math.ldexp(estimate,-exponent)

# demo
for number in np.linspace(-2,12,15):
    recip = reciprocal_by_mult(number)
    print('Reciprocal of',number,'=',recip,', whose reciprocal is',reciprocal_by_mult(recip))

Let's now search for zeros of any polynomial *f*. We'll start by importing code from Level 4 Part 1.

In [ ]:
def coef_poly(coef_list,power=None):
    if power==None: 
        return np.array(coef_list)
    coef_full = np.zeros(max(power)+1)
    coef_full[power] = coef_list
    return coef_full

def eval_poly(x,coef):
    poly = coef[-1] 
    for c in coef[-2::-1]:
        poly = poly * x  + c
    return poly

coef_list,power = [2,-8,1],[0,2,4]
x = np.array([0,1,2,3])
coef = coef_poly(coef_list,power)
print('coef =',coef,'\nx =',x,'\ny =',eval_poly(x,coef))
# plot poly
x = np.linspace(-3,3,100)
y = eval_poly(x,coef)
plt.title("Plot of Polynomial")
plt.plot(x,y,x,np.zeros(100))
plt.show()

The bisection method starts with an interval that we know contains a zero 
since *f* has different signs at the endpoints. 
We then calculate *f* for the midpoint, use that to halve the interval, and repeat. 
This is guaranteed to find a zero but will take many steps 
if the starting interval is much wider than the desired precision.

In [ ]:
def bisection_poly(x1,x2,coef,tol=1E-14):
    y1 = eval_poly(x1,coef)
    if y1 * eval_poly(x2,coef) > 0:
        print('[x1 x2] = [',x1,x2,'] is not a good starting interval')
        return np.nan,0
    for n in range(1000):
        mid = 0.5 * (x1 + x2)
        ymid = eval_poly(mid,coef)
        if abs(x2-mid) < tol:
            break
        elif ymid * y1 > 0:  
            x1 = mid
        else:
            x2 = mid
    return mid,n 

coef = [ 2, 0, -8, 0,  1] 
x2=1
for x1 in (1000,10,4,2,0,-3,-100):
    val_zero,n = bisection_poly(x1,x2,coef)
    print('For [x1 x2] = [',x1,x2,'], zero at',val_zero,'discovered in',n,'iterations')

Here's one way to look for values bracketing a zero. 
It starts with the end points and then cycles through the interval 
using a gradually narrowing grid. 
Once it finds bracketing values, it returns the tightest interval it found.

In [ ]:
def seed_bisection(xmin,xmax,coef,max_iter=1000):
    ymin = eval_poly(xmin,coef)
    ymax = eval_poly(xmax,coef)
    if ymin * ymax <= 0:
        return xmin,xmax
    sample = [xmin, xmax]
    t = dt = 1/2
    for n in range(2,max_iter):
        x = xmin + t * (xmax - xmin)
        sample.append(x)
        y = eval_poly(x,coef)
        if y * ymax <= 0:
            sample = np.array(sample)
            nearest_high = np.min(sample[sample>x])
            nearest_low = np.max(sample[sample<x])
            if nearest_high - x < x - nearest_low:
                return x,nearest_high
            else:
                return nearest_low,x
        dt *= 1 - 1/n       # reduce dt by slowly declining fraction
        t = (t + dt) % 1    # retain only the fractional part of t
    return np.nan,xmax      # np.nan flags the failure to bracket zero

coef = [ 2, 0, -8, 0,  1] 
for xmin in (-100,-10,-5,0,1,5):
    x1,x2 = seed_bisection(xmin,100,coef)
    if np.isnan(x1):     # checks for np.nan
        print('Could not find values on both side of zero between',xmin,'and',x2)
    else:
        print('for xmin =',xmin,'confirm zero between',x1,'and',x2)

Given any two points (*x*1,*y*1) and (*x*2,*y*2) with different *y* values, 
the secant method chooses the next point assuming that *y* is linear in *x*. 
It usually works much faster than bisection but is not guaranteed to converge.

In [ ]:
def secant_poly(x1,x2,coef,tol=1E-14):
    y1 = eval_poly(x1,coef)
    y2 = eval_poly(x2,coef)
    for n in range(100):
        x3 = x1 - y1 * (x2 - x1) / (y2 - y1)
        y3 = eval_poly(x3,coef)
        if abs(y3) < tol: 
            break 
        if y3 != y2:
            x1,y1 = x2,y2
        x2,y2 = x3,y3
    return x3,n 

coef_list,power = [2,-8,1],[0,2,4]
coef = coef_poly(coef_list,power)
zero,n = secant_poly(10,2,coef)
print('zero at',zero,'identified in',n,'iterations')

When the two points are close, the secant approximates the tangent line. 
The Newton-Raphson method works with a single point at a time and its derivative. 
Like the secant method, it is not guaranteed to converge. 
When the derivative is zero, the code here adjusts *x* to keep the update finite.

In [ ]:
def deriv_polynomial(coef,order=1):
    for i in range(order):  
        coef = [0] if len(coef)==1 else coef[1:] * np.arange(1,len(coef))
    return coef

def newton_raphson_poly(x0,coef,tol=1E-14):
    x = x0
    coef_deriv = deriv_polynomial(coef,1)
    for n in range(100):
        y = eval_poly(x,coef)
        if abs(y) < tol:
            break
        deriv = eval_poly(x,coef_deriv)
        if deriv == 0:
            x += np.sqrt(tol)  # move x slightly
            deriv = eval_poly(x,coef_deriv)
        x -= y / deriv
    return x,n

coef_list,power = [2,-8,1],[0,2,4]
coef = coef_poly(coef_list,power)
for x0 in (-5,-1,0,4,100):
    x,n = newton_raphson_poly(x0,coef)
    print('zero at',x,'discovered in',n,'iterations')

We can also use Newton-Raphson to solve for stationary points, where the derivative is zero.

In [ ]:
def stationary_point(x0,coef,tol=1E-14):
    coef_deriv = deriv_polynomial(coef,1)
    xx,n = newton_raphson_poly(x0,coef_deriv,tol)
    coef_deriv2 = deriv_polynomial(coef_deriv,1)
    deriv2 = eval_poly(xx,coef_deriv2)
    return xx,n,deriv2

# test
coef_list,power = [2,-8,1],[0,2,4]
coef = coef_poly(coef_list,power)
for x0 in (-5,-1,0.5,4,100):
    xx,n,deriv2 = stationary_point(x0,coef)
    str1 = 'stationary point at '+str(xx)+' discovered in '+str(n)+' iterations'
    if deriv2 < 0:
        print(str1,'and is local maximum')
    elif deriv2 > 0:
        print(str1,'and is local minimum')
    else:
        print(str1,'and second derivative is zero')